# 📉 Обучение модели прогноза Error Rate

Загрузка датасета, обучение Ridge-регрессии (горизонт 5 мин) и сохранение модели.


## 1. Настройка окружения


In [1]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import json
import joblib
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

FREQ_SEC   = 15
DATA_PATH  = '../data/error_rate_dataset.csv'

print('✅ Окружение готово')


✅ Окружение готово


## 2. Загрузка и обзор датасета


In [2]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['datetime'] = pd.to_datetime(df_raw['timestamp'], unit='s')
df_raw = df_raw.sort_values('datetime').reset_index(drop=True)

# Полный индекс 15 с
full_idx = pd.date_range(df_raw['datetime'].iloc[0],
                         df_raw['datetime'].iloc[-1], freq='15s')
df = (df_raw.drop_duplicates('timestamp')
            .set_index('datetime')[['error_rate']]
            .reindex(full_idx))
df.index.name = 'datetime'
gaps = df['error_rate'].isna().sum()
df['error_rate'] = df['error_rate'].interpolate('time').fillna(0.0)
# error_rate ∈ [0, 1]; защищаемся от выбросов
df['error_rate'] = df['error_rate'].clip(0.0, 1.0)

er = df['error_rate']
print(f"Строк: {len(df):,}  |  Пропусков заполнено: {gaps}")
print(f"Период: {df.index[0]}  →  {df.index[-1]}")
print(f"error_rate: mean={er.mean():.5f}  std={er.std():.5f}  "
      f"min={er.min():.5f}  max={er.max():.5f}")


Строк: 30,134  |  Пропусков заполнено: 0
Период: 2026-05-25 18:42:00  →  2026-05-31 00:15:15
error_rate: mean=0.02988  std=0.02326  min=0.01202  max=0.42189


In [3]:
# ── Признаки для Ridge ────────────────────────────────────────────────────
def make_features(series):
    f = pd.DataFrame(index=series.index)
    for lag in [1, 2, 4, 8, 16, 32, 60, 120, 240]:
        f[f'lag_{lag}'] = series.shift(lag)
    for w in [4, 20, 60, 120, 240]:
        f[f'roll_mean_{w}'] = series.shift(1).rolling(w).mean()
        f[f'roll_std_{w}']  = series.shift(1).rolling(w).std()
    for span in [4, 20, 60]:
        f[f'ewm_{span}'] = series.shift(1).ewm(span=span).mean()
    f['hour_sin'] = np.sin(2*np.pi*series.index.hour/24)
    f['hour_cos'] = np.cos(2*np.pi*series.index.hour/24)
    f['minute_sin'] = np.sin(2*np.pi*series.index.minute/60)
    f['minute_cos'] = np.cos(2*np.pi*series.index.minute/60)
    f['diff_1']  = series.diff(1).shift(1)
    f['diff_20'] = series.diff(20).shift(1)
    return f


print("✅ Функция make_features готова")


✅ Функция make_features готова


## 3. Обучение модели и сохранение


In [4]:
# ── Гиперпараметры финальной модели ────────────────────────────────────
FINAL_HORIZON_MIN = 5
FINAL_HORIZON_STEPS = int(FINAL_HORIZON_MIN * 60 / FREQ_SEC)   # 20 шагов
LAGS         = [1, 2, 4, 8, 16, 32, 60, 120, 240]
ROLL_WINDOWS = [4, 20, 60, 120, 240]
EWM_SPANS    = [4, 20, 60]
ALPHA_RIDGE  = 10.0

# Минимальное число точек истории, нужное для построения полного вектора признаков
# (диктуется самым большим лагом / окном; +1 запас на shift(1))
MIN_HISTORY_POINTS = max(max(LAGS), max(ROLL_WINDOWS)) + 1   # 241

FEATURE_COLS = (
    [f'lag_{l}' for l in LAGS]
    + [f'roll_mean_{w}' for w in ROLL_WINDOWS]
    + [f'roll_std_{w}'  for w in ROLL_WINDOWS]
    + [f'ewm_{s}' for s in EWM_SPANS]
    + ['hour_sin', 'hour_cos', 'minute_sin', 'minute_cos', 'diff_1', 'diff_20']
)

print(f"Горизонт прогноза: {FINAL_HORIZON_MIN} мин = {FINAL_HORIZON_STEPS} шагов")
print(f"Признаков: {len(FEATURE_COLS)}")
print(f"Требуется истории: {MIN_HISTORY_POINTS} точек ({MIN_HISTORY_POINTS*FREQ_SEC/60:.1f} мин)")


Горизонт прогноза: 5 мин = 20 шагов
Признаков: 28
Требуется истории: 241 точек (60.2 мин)


In [5]:
# ── Финальное обучение Ridge на всей истории ───────────────────────────
# Используем уже определённую функцию make_features из секции 10
X_full = make_features(er)
y_full = er.shift(-FINAL_HORIZON_STEPS).rename('target')

missing = set(FEATURE_COLS) - set(X_full.columns)
extra = set(X_full.columns) - set(FEATURE_COLS)

assert not missing, f"Отсутствуют признаки: {missing}"
assert not extra, f"Лишние признаки: {extra}"

X_full = X_full[FEATURE_COLS]

data_full = pd.concat([X_full, y_full], axis=1).dropna()
print(f"Размер обучающей выборки после dropna: {len(data_full):,} строк")

X_train = data_full[FEATURE_COLS].values
y_train = data_full['target'].values

final_scaler = StandardScaler()
X_train_scaled = final_scaler.fit_transform(X_train)

final_ridge = Ridge(alpha=ALPHA_RIDGE)
final_ridge.fit(X_train_scaled, y_train)

# Оценка на in-sample (только для контроля; честные метрики уже даны walk-forward)
y_pred_in = final_ridge.predict(X_train_scaled)
mse_in  = mean_squared_error(y_train, y_pred_in)
mae_in  = mean_absolute_error(y_train, y_pred_in)
mask_nz = y_train != 0
mape_in = (np.mean(np.abs((y_pred_in[mask_nz] - y_train[mask_nz]) / y_train[mask_nz])) * 100
           if mask_nz.any() else np.nan)
wape_in = (np.sum(np.abs(y_pred_in - y_train)) / np.sum(np.abs(y_train)) * 100
           if np.sum(np.abs(y_train)) > 0 else np.nan)
print(f"\nIn-sample:   MSE={mse_in:.6f}  MAE={mae_in:.6f}  MAPE={mape_in:.2f}%  WAPE={wape_in:.2f}%")


Размер обучающей выборки после dropna: 29,874 строк

In-sample:   MSE=0.000257  MAE=0.005002  MAPE=14.39%  WAPE=16.73%


In [6]:
# ── Сохранение модели и конфига для прод-сервиса ────────────────────────
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH  = MODELS_DIR / 'error_rate_forecast_model.joblib'
CONFIG_PATH = MODELS_DIR / 'error_rate_model_config.json'

# Единый артефакт: модель + scaler + список признаков + метаданные
bundle = {
    'model': final_ridge,
    'scaler': final_scaler,
    'feature_cols': FEATURE_COLS,
    'meta': {
        'model_type': 'ridge',
        'metric_type': 'error_rate',
        'forecast_horizon_steps': FINAL_HORIZON_STEPS,
        'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
        'scrape_interval_sec': FREQ_SEC,
        'lags': LAGS,
        'rolling_windows': ROLL_WINDOWS,
        'ewm_spans': EWM_SPANS,
        'min_history_points': MIN_HISTORY_POINTS,
        'alpha': ALPHA_RIDGE,
    },
}
joblib.dump(bundle, MODEL_PATH)
print(f"✅ Модель сохранена: {MODEL_PATH.resolve()}")

# Конфиг для приложения (читается app/config.py)
config_json = {
    'model_type': 'ridge',
    'feature_cols': FEATURE_COLS,
    'categorical_features': [],
    'forecast_horizon_steps': FINAL_HORIZON_STEPS,
    'forecast_horizon_seconds': FINAL_HORIZON_STEPS * FREQ_SEC,
    'scrape_interval_sec': FREQ_SEC,
    'lags': LAGS,
    'rolling_windows': ROLL_WINDOWS,
    'ewm_spans': EWM_SPANS,
    'min_history_points': MIN_HISTORY_POINTS,
    'alpha': ALPHA_RIDGE,
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config_json, f, indent=2, ensure_ascii=False)
print(f"✅ Конфиг сохранён:  {CONFIG_PATH.resolve()}")

# Контрольная проверка: загрузка и предсказание на последней точке
loaded = joblib.load(MODEL_PATH)
last_row = data_full[FEATURE_COLS].iloc[[-1]].values
last_scaled = loaded['scaler'].transform(last_row)
last_pred = float(loaded['model'].predict(last_scaled)[0])
print(f"\nКонтрольная загрузка: предсказание для последней точки = {last_pred:.5f} error_rate")


✅ Модель сохранена: /Users/anastasiagusak/Documents/4 курс/MLPredictor/models/error_rate_forecast_model.joblib
✅ Конфиг сохранён:  /Users/anastasiagusak/Documents/4 курс/MLPredictor/models/error_rate_model_config.json

Контрольная загрузка: предсказание для последней точки = 0.02573 error_rate
